# 05 — Huấn luyện, Đánh giá và Trích xuất Embedding: Mô hình SASRec

Notebook này thực hiện trọn vẹn quy trình huấn luyện và đánh giá mô hình **SASRec** (*Self-Attentive Sequential Recommendation — Kang & McAuley, ICDM 2018*) trên bộ dữ liệu **Diginetica** với **2 random seed (`2024`, `2025`)**.

## Đặc trưng kiến trúc của SASRec và Giao thức thí nghiệm

1. **Kiến trúc Causal Multi-Head Self-Attention (Transformer):**
   - Kết hợp `item_embedding` và `position_embedding` (`max_seq_length = 50`), đi qua `LayerNorm` và `Dropout`.
   - Đi qua `n_layers = 2` khối Transformer với mặt nạ nhân quả (**Causal Mask** — vị trí $t$ chỉ được nhìn các vị trí $\le t$) và Point-wise Feed-Forward (`inner_size = 256`, hàm kích hoạt `GELU`).
   - Lấy vector biểu diễn tại vị trí cuối thực tế của phiên nhân vô hướng với toàn bộ bảng `item_embedding`.
2. **Đảm bảo tính công bằng tuyệt đối với GRU4Rec và NARM:**
   - Dùng chung tập `train_examples.parquet`, `validation_examples.parquet`, `test_examples.parquet`, `item_mapping.csv`.
   - Khóa cứng gradient và mask điểm (`-1e9`) đối với `padding_idx = 0` và `7.647` Controlled Cold items.
   - Lưu ý khi trích xuất embedding: Chỉ trích xuất đúng `model.item_embedding.weight[1:]` (`37.583` hàng), tuyệt đối không nhầm với `position_embedding`.
   - Xuất đủ 8 file chuẩn vào `runs/sasrec/seed_{2024,2025}/` và bảng cấp độ item vào `results/item_level/sasrec_seed{2024,2025}.parquet`.

## 1. Thiết lập môi trường, thiết bị (GPU/CPU) và đọc cấu hình

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "BTL-K03-tong-quan-de-tai.md").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án BTL-K03.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from btl_k03.paths import FIGURES_DIR, ITEM_LEVEL_DIR, PROCESSED_DIGI_DIR, RUNS_DIR, TABLES_DIR, ensure_output_dirs
from btl_k03.utils import load_experiment_config, load_item_mapping, set_global_seed
from btl_k03.models import SessionPrefixDataset, build_model_from_config, train_and_evaluate_seed

ensure_output_dirs()

MODEL_NAME = "sasrec"
CONFIG = load_experiment_config(MODEL_NAME)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python executable : {sys.executable}")
print(f"PyTorch version   : {torch.__version__}")
print(f"Device            : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Model architecture: {CONFIG['architecture']}")
print(f"Seeds             : {CONFIG['seeds']}")
print(f"Model params      : {json.dumps(CONFIG['model_params'], ensure_ascii=False)}")
print(f"Training config   : {json.dumps(CONFIG['training'], ensure_ascii=False)}")

## 2. Kiểm tra dữ liệu đầu vào và mặt nạ Warm / Controlled Cold (`Option B`)

In [ ]:
item_mapping = load_item_mapping(PROCESSED_DIGI_DIR)
train_examples = pd.read_parquet(PROCESSED_DIGI_DIR / "train_examples.parquet")
val_examples = pd.read_parquet(PROCESSED_DIGI_DIR / "validation_examples.parquet")
test_examples = pd.read_parquet(PROCESSED_DIGI_DIR / "test_examples.parquet")
test_last_examples = pd.read_parquet(PROCESSED_DIGI_DIR / "test_last_item_examples.parquet")

n_total_catalog = len(item_mapping)
n_warm = int(item_mapping["is_trainable_warm"].sum())
n_cold = int(item_mapping["is_controlled_cold"].sum())

summary_input = pd.DataFrame([
    {"Chỉ tiêu": "Tổng số item trong bảng Embedding (ID 1..N)", "Giá trị": f"{n_total_catalog:,}"},
    {"Chỉ tiêu": "Số item Warm hợp lệ (is_trainable_warm = True)", "Giá trị": f"{n_warm:,}"},
    {"Chỉ tiêu": "Số item Controlled Cold 20% (is_controlled_cold = True)", "Giá trị": f"{n_cold:,}"},
    {"Chỉ tiêu": "Số mẫu Train (All-prefix)", "Giá trị": f"{len(train_examples):,}"},
    {"Chỉ tiêu": "Số mẫu Validation Warm (All-prefix)", "Giá trị": f"{len(val_examples):,}"},
    {"Chỉ tiêu": "Số mẫu Test Warm chính (All-prefix)", "Giá trị": f"{len(test_examples):,}"},
    {"Chỉ tiêu": "Số mẫu Test Warm phiên cuối (Last-item)", "Giá trị": f"{len(test_last_examples):,}"},
])
display(summary_input)

## 3. Khởi tạo kiến trúc SASRec và Sanity Check khóa Gradient Cold Item

In [ ]:
set_global_seed(2024)
sanity_model = build_model_from_config(CONFIG, item_mapping, DEVICE)
print(sanity_model)

sample_ds = SessionPrefixDataset(train_examples.head(64), max_seq_length=CONFIG["max_seq_length"])
sample_seq = sample_ds.sequences.to(DEVICE)
sample_len = sample_ds.lengths.to(DEVICE)
sample_target = sample_ds.targets_tensor.to(DEVICE)

sanity_model.zero_grad()
logits = sanity_model.full_sort_logits(sample_seq, sample_len, mask_cold_and_pad=True)
loss = torch.nn.functional.cross_entropy(logits, sample_target)
loss.backward()

grad_matrix = sanity_model.item_embedding.weight.grad.detach().cpu().numpy()
cold_ids = item_mapping.loc[item_mapping["is_controlled_cold"], "internal_item_id"].to_numpy()
warm_ids = item_mapping.loc[item_mapping["is_trainable_warm"], "internal_item_id"].to_numpy()

max_cold_grad = float(np.max(np.abs(grad_matrix[cold_ids])))
max_pad_grad = float(np.max(np.abs(grad_matrix[0])))
max_warm_grad = float(np.max(np.abs(grad_matrix[warm_ids])))

print(f"\n[SANITY CHECK GRADIENT - SASRec]")
print(f"- Loss khởi tạo (Cross-Entropy trên 29,936 warm items): {loss.item():.4f} (Lý thuyết ~ ln(29936) = {np.log(n_warm):.4f})")
print(f"- Max |grad| của hàng Padding (ID 0)                  : {max_pad_grad:.10f}")
print(f"- Max |grad| của 7,647 hàng Controlled Cold           : {max_cold_grad:.10f}")
print(f"- Max |grad| của 29,936 hàng Trainable Warm           : {max_warm_grad:.10f}")
assert max_cold_grad == 0.0 and max_pad_grad == 0.0 and max_warm_grad > 0.0
del sanity_model

## 4. Huấn luyện SASRec trên 2 Seed (`2024` và `2025`)

In [ ]:
MAX_EPOCHS_OVERRIDE = None  # Đặt bằng None để chạy full theo config (30 epochs + Early Stopping)

run_outputs = {}
for seed in CONFIG["seeds"]:
    run_outputs[seed] = train_and_evaluate_seed(
        model_name=MODEL_NAME,
        seed=seed,
        config=CONFIG,
        processed_dir=PROCESSED_DIGI_DIR,
        max_epochs_override=MAX_EPOCHS_OVERRIDE,
    )

## 5. Tổng hợp kết quả chất lượng gợi ý (Warm-item Accuracy vs. Cold-start Impact)

In [ ]:
acc_rows = []
stratified_frames = []
cold_diag_rows = []

for seed, out in run_outputs.items():
    tm = out["test_metrics"]
    w_all = tm["test_warm_all_prefix"]
    w_last = tm["test_warm_last_item"]
    acc_rows.append({
        "model": MODEL_NAME.upper(),
        "seed": seed,
        "best_epoch": tm["best_epoch"],
        "val_best_mrr@20": tm["best_validation_score"],
        "test_warm_recall@20": w_all["recall@20"],
        "test_warm_mrr@20": w_all["mrr@20"],
        "test_warm_ndcg@20": w_all["ndcg@20"],
        "test_last_recall@20": w_last["recall@20"],
        "test_last_mrr@20": w_last["mrr@20"],
        "test_last_ndcg@20": w_last["ndcg@20"],
        "cold_max_drift": tm["cold_embedding_max_abs_drift"],
    })
    strat_df = out["test_stratified_df"].copy()
    strat_df.insert(0, "seed", seed)
    stratified_frames.append(strat_df)
    cold_diag_rows.append({"seed": seed, **tm["cold_start_diagnostic"]})

acc_df = pd.DataFrame(acc_rows)
print("=== BẢNG 1: CHẤT LƯỢNG DỰ ĐOÁN CỦA SASRec TRÊN TẬP TEST WARM ===")
display(acc_df)

print("\n=== BẢNG 2: ĐỘ CHÍNH XÁC TÁCH THEO NHÓM ĐỘ PHỔ BIẾN CỦA TARGET (POPULAR / MEDIUM / RARE) ===")
display(pd.concat(stratified_frames, ignore_index=True))

print("\n=== BẢNG 3: ĐỐI CHỨNG HIỆU NĂNG KHI BAO GỒM 20% CONTROLLED COLD & NATURAL COLD TRONG TEST FULL ===")
display(pd.DataFrame(cold_diag_rows))

## 6. Chỉ số Thiên lệch độ phổ biến ở đầu ra (`Coverage@20`, `Gini@20`, `ARP@20`)

In [ ]:
bias_rows = []
exposure_frames = []

for seed, out in run_outputs.items():
    pb = out["test_metrics"]["popularity_bias"]
    bias_rows.append({
        "model": MODEL_NAME.upper(),
        "seed": seed,
        "coverage_warm@20": pb["coverage_warm@20"],
        "coverage_full@20": pb["coverage_full@20"],
        "gini_warm@20": pb["gini_warm@20"],
        "gini_full@20": pb["gini_full@20"],
        "arp@20": pb["arp@20"],
        "arp_log1p@20": pb["arp_log1p@20"],
        "popular_exposure_pct": pb["popular_group_exposure_share_pct"],
        "rare_exposure_pct": pb["rare_group_exposure_share_pct"],
    })
    exp_df = out["group_exposure_df"].copy()
    exp_df.insert(0, "seed", seed)
    exposure_frames.append(exp_df)

print("=== BẢNG 4: CHỈ SỐ POPULARITY BIAS ĐẦU RA CỦA SASRec ===")
display(pd.DataFrame(bias_rows))

print("\n=== BẢNG 5: PHÂN BỔ LƯỢT HIỂN THỊ TOP-20 THEO NHÓM POPULAR / MEDIUM / RARE / COLD ===")
display(pd.concat(exposure_frames, ignore_index=True))

## 7. Biểu đồ Learning Curve, Phân bố Norm Embedding và Kiểm tra các file bàn giao

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for seed, out in run_outputs.items():
    hist = out["history_df"]
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=f"Seed {seed}")
    axes[1].plot(hist["epoch"], hist["validation_mrr@20"], marker="s", label=f"Seed {seed} (MRR@20)")
    axes[1].plot(hist["epoch"], hist["validation_recall@20"], linestyle="--", label=f"Seed {seed} (Recall@20)")

axes[0].set_title("SASRec — Training Cross-Entropy Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title("SASRec — Validation Recall@20 & MRR@20")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()

first_seed = CONFIG["seeds"][0]
il_df = run_outputs[first_seed]["item_level_df"]
sns.boxplot(
    data=il_df,
    x="popularity_group",
    y="embedding_norm",
    order=["popular", "medium", "rare", "cold"],
    palette="Set2",
    ax=axes[2],
)
axes[2].set_title(f"SASRec (Seed {first_seed}) — L2 Norm theo nhóm Popularity")
axes[2].set_xlabel("Nhóm Item")
axes[2].set_ylabel("L2 Norm ||e_i||_2")

plt.tight_layout()
plt.show()

REQUIRED_FILES = [
    "config.yaml",
    "checkpoint.pth",
    "training.log",
    "training_history.csv",
    "test_metrics.json",
    "top20.parquet",
    "item_embeddings.npy",
    "item_embedding_mapping.csv",
]

manifest_rows = []
for seed, out in run_outputs.items():
    rdir = out["run_dir"]
    for fname in REQUIRED_FILES:
        fpath = rdir / fname
        manifest_rows.append({
            "seed": seed,
            "file": fname,
            "exists": fpath.is_file(),
            "size_kb": round(fpath.stat().st_size / 1024.0, 2) if fpath.is_file() else 0.0,
        })

manifest_df = pd.DataFrame(manifest_rows)
assert manifest_df["exists"].all(), "Thiếu file bàn giao trong runs/sasrec!"
display(manifest_df)